# Chapter 12 — Public Finance

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch12_public_finance.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 12.1 Budget documents as data

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from pypdf import PdfReader

BUDGET_DIR = Path("data/raw/budget")
HEADERS = {"User-Agent": "python-for-indian-economics (teaching use)"}
SITE = "https://www.indiabudget.gov.in/"


def budget_pdf(path_on_site, name):
    """Download a Union Budget PDF once; return the local file path."""
    local = BUDGET_DIR / f"{name}.pdf"
    if not local.exists():
        reply = requests.get(SITE + path_on_site, headers=HEADERS, timeout=120)
        reply.raise_for_status()
        if reply.content[:4] != b"%PDF":
            raise ValueError(f"{path_on_site} did not return a PDF")
        BUDGET_DIR.mkdir(parents=True, exist_ok=True)
        local.write_bytes(reply.content)
    return local


def pdf_text(path):
    """All the text pypdf can find in a PDF, page after page."""
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def archive(budget_year):
    """Folder of a Budget on the site: the latest is at the top level."""
    return "" if budget_year == "2026-27" else f"budget{budget_year}/"


bag_file = budget_pdf("doc/Budget_at_Glance/bag1.pdf", "bag1_2026-27")
bag_text = pdf_text(bag_file)
print(len(PdfReader(bag_file).pages), "pages,", len(bag_text), "characters")

## 12.2 What the deficits mean

In [ ]:
flat = re.sub(r"\s+", " ", bag_text)
definition = re.search(r"Fiscal Deficit \(FD\) is the difference.*?interest payments\.",
                       flat)
print(definition.group(0))

## 12.3 Reading a table from a PDF

In [ ]:
start = bag_text.find("17. Fiscal Deficit")
print(bag_text[start: start + 120])

In [ ]:
ROWS = {"interest": r"\d+\.\s*Interest Payments",
        "fiscal_deficit": r"\d+\.\s*Fiscal Deficit",
        "revenue_deficit": r"\d+\.\s*Revenue Deficit",
        "primary_deficit": r"\d+\.\s*Primary Deficit"}


def bag_row(text, label):
    """The four Rs crore values and the % of GDP values after a row label."""
    for m in re.finditer(label, text):
        window = text[m.end(): m.end() + 250]
        window = re.sub(r"\[[^\]]*\]|\(\d+-\d+\)", " ", window)
        crore = re.findall(r"\b\d{5,}\b", window)
        if len(crore) >= 4:
            pct = re.findall(r"\((\d+\.\d+)\)", window)
            return [int(x) for x in crore[:4]], [float(x) for x in pct[:4]]
    raise ValueError(f"row {label!r} not found")


for item, label in ROWS.items():
    print(f"{item:16}", bag_row(bag_text, label))

## 12.4 Eight Budgets in a loop

In [ ]:
def fy(start):
    return f"{start}-{str(start + 1)[2:]}"


budget_years = [fy(y) for y in range(2019, 2027)]
records = []
for b in budget_years:
    text = pdf_text(budget_pdf(archive(b) + "doc/Budget_at_Glance/bag1.pdf",
                               f"bag1_{b}"))
    parsed = {item: bag_row(text, label) for item, label in ROWS.items()}
    fd, ip, pdf_ = (np.array(parsed[k][0]) for k in
                    ("fiscal_deficit", "interest", "primary_deficit"))
    assert (fd - ip == pdf_).all(), f"identity fails in Budget {b}"
    first = int(b[:4])
    columns = [(fy(first - 2), "Actuals"), (fy(first - 1), "BE"),
               (fy(first - 1), "RE"), (fy(first), "BE")]
    for j, (year, kind) in enumerate(columns):
        row = {"budget": b, "year": year, "kind": kind}
        for item in ROWS:
            row[item] = parsed[item][0][j]
            if item != "interest":
                row[item + "_pct"] = parsed[item][1][j]
        records.append(row)

bag = pd.DataFrame(records)
print(len(bag), "rows from", bag["budget"].nunique(), "Budgets; identity holds in all")
print(bag.columns.tolist())

In [ ]:
fd_vintages = (bag.pivot_table(index="year", columns="kind",
                               values="fiscal_deficit", aggfunc="last")
                  [["BE", "RE", "Actuals"]])
fd_vintages["Actual vs BE, %"] = (fd_vintages["Actuals"] / fd_vintages["BE"] - 1) * 100
fd_vintages.dropna().round(1)

In [ ]:
actuals = bag[bag["kind"] == "Actuals"].drop_duplicates("year", keep="last")
latest = bag[(bag["budget"] == "2026-27") & (bag["kind"].isin(["RE", "BE"]))
             & (bag["year"].isin(["2025-26", "2026-27"]))]
latest = latest[~((latest["year"] == "2025-26") & (latest["kind"] == "BE"))]
fiscal = (pd.concat([actuals, latest])
            .set_index("year")
            .drop(columns="budget"))
fiscal[["kind", "fiscal_deficit", "revenue_deficit", "primary_deficit",
        "interest"]]

## 12.5 Ratios to GDP, and a revision check

In [ ]:
from indiadata import mospi_nas

gdp = mospi_nas("GDP").set_index("year")["current_price"]   # Rs crore

note = re.search(r"GDP for FY 2026-27 is estimated at\s*₹([\d,]+)\s*crore", flat)
gdp_be = int(note.group(1).replace(",", ""))
print("Budget's GDP assumption for 2026-27:", note.group(1), "=", gdp_be, "crore")
print("MoSPI GDP 2025-26 (advance estimate):", gdp["2025-26"])
print(f"Implied nominal growth: {gdp_be / gdp['2025-26'] - 1:.1%}")
gdp["2026-27"] = gdp_be

In [ ]:
for item in ["fiscal_deficit", "revenue_deficit", "primary_deficit", "interest"]:
    fiscal[item + "_gdp"] = fiscal[item] / gdp.reindex(fiscal.index) * 100

check = pd.DataFrame({"ours": fiscal["fiscal_deficit_gdp"].round(2),
                      "published": fiscal["fiscal_deficit_pct"]})
check["gap"] = (check["ours"] - check["published"]).round(2)
check

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE, LIGHT = "#1f4e79", "#c55a11", "#9db7d5"
x = np.arange(len(fiscal))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(x, fiscal["fiscal_deficit_gdp"], marker="o", color=BLUE,
        linewidth=2, label="Fiscal deficit")
ax.plot(x, fiscal["revenue_deficit_gdp"], marker="o", color=LIGHT,
        linewidth=2, label="Revenue deficit")
ax.plot(x, fiscal["primary_deficit_gdp"], marker="o", color=ORANGE,
        linewidth=2, label="Primary deficit")
ax.plot(x, fiscal["interest_gdp"], color="grey", linestyle="--",
        label="Interest payments")
ax.axvspan(len(fiscal) - 2.5, len(fiscal) - 0.5, color="grey", alpha=0.1)
ax.text(len(fiscal) - 2.4, 8.8, "RE / BE", fontsize=9, color="grey")
ax.set_xticks(x, fiscal.index, rotation=45)
ax.set_ylabel("% of GDP")
ax.set_title("Union government deficits, 2017-18 to 2026-27", loc="left",
             fontsize=13)
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Sources: Budget at a Glance 2019-20 to 2026-27 "
         "(indiabudget.gov.in); GDP from MoSPI NAS, 2026-27 as assumed in the "
         "Budget.", fontsize=8, color="grey")
plt.show()

In [ ]:
fiscal[["interest_gdp", "primary_deficit_gdp"]].iloc[[0, -2, -1]].round(2)

## 12.6 The stock of debt

In [ ]:
liab_records = []
for b in ["2019-20", "2021-22", "2023-24", "2025-26", "2026-27"]:
    text = pdf_text(budget_pdf(archive(b) + "doc/rec/annex91.pdf",
                               f"liabilities_{b}"))
    if len(text) < 100:
        print(f"Budget {b}: no text in the PDF (a scanned image) - skipped")
        continue
    years = re.search(r"1950-(?:51|1951)((?:\s+\d{4}-\d{2,4}){6})", text).group(1).split()
    totals = re.search(r"Total - Liabilities\s+[\d.]+((?:\s+[\d.]+){6})",
                       text).group(1).split()
    kinds = ["Actuals"] * 4 + ["RE", "BE"]
    for y, v, k in zip(years, totals, kinds):
        liab_records.append({"budget": b, "year": fy(int(y[:4])), "kind": k,
                             "liabilities": float(v)})

liab = pd.DataFrame(liab_records)
print(len(liab), "values from", liab["budget"].nunique(), "statements")

In [ ]:
acts = liab[liab["kind"] == "Actuals"]
spread = acts.groupby("year")["liabilities"].agg(["count", "min", "max"])
spread["max - min"] = spread["max"] - spread["min"]
spread[spread["count"] > 1]

In [ ]:
debt = acts.drop_duplicates("year", keep="last").set_index("year")["liabilities"]
last = liab[liab["budget"] == "2026-27"].set_index("year")["liabilities"]
debt["2025-26"], debt["2026-27"] = last["2025-26"], last["2026-27"]
debt = debt.sort_index()
debt_ratio = debt / gdp.reindex(debt.index) * 100
pd.DataFrame({"liabilities, Rs lakh crore": (debt / 1e5).round(1),
              "% of GDP": debt_ratio.round(1)})

In [ ]:
full_bag = pdf_text(budget_pdf("doc/Budget_at_Glance/budget_at_a_glance.pdf",
                               "bag_full_2026-27"))
after = full_bag[full_bag.find("TRENDS IN DEBT"):]
values = [float(v) for v in re.findall(r"(\d+\.\d)%", after)]
labels = [v for v in values if v > 40][:20]      # debt labels; deficits are < 10
chart_years = [fy(y) for y in range(2017, 2027)]
published = pd.Series(labels[:10], index=chart_years)   # Statement of Liabilities
frbm = pd.Series(labels[10:], index=chart_years)        # FRBM Act definition
compare = pd.DataFrame({"ours": debt_ratio.reindex(chart_years).round(1),
                        "published": published, "FRBM def.": frbm})
compare["gap"] = (compare["ours"] - published).round(1)
compare["FRBM - published"] = (frbm - published).round(1)
print(f"FRBM debt minus Statement-of-Liabilities debt: "
      f"{compare['FRBM - published'].min():.1f} to "
      f"{compare['FRBM - published'].max():.1f} points, "
      f"mean {compare['FRBM - published'].mean():.1f}")
compare

## 12.7 The debt-dynamics equation

## 12.8 Measuring *i* and *g* for India

In [ ]:
dd = pd.DataFrame({"debt": debt, "gdp": gdp.reindex(debt.index)})
dd = dd.join(fiscal[["interest", "primary_deficit"]])
dd["i"] = dd["interest"] / dd["debt"].shift(1)
dd["g"] = dd["gdp"] / dd["gdp"].shift(1) - 1
dd["i_minus_g"] = dd["i"] - dd["g"]
dd["d"] = dd["debt"] / dd["gdp"]
dd["pd"] = dd["primary_deficit"] / dd["gdp"]
(dd[["i", "g", "i_minus_g"]].dropna() * 100).round(2)

In [ ]:
hist = dd.loc["2017-18":"2024-25"]
print(f"Average effective interest rate, 2017-18 to 2024-25: {hist['i'].mean():.2%}")
print(f"Average nominal GDP growth,      2017-18 to 2024-25: {hist['g'].mean():.2%}")
print(f"Years with i < g: {(hist['i_minus_g'] < 0).sum()} of {len(hist)}")

## 12.9 Accounting for the change in the debt ratio

In [ ]:
dec = pd.DataFrame(index=hist.index)
dec["change in d"] = dd["d"].diff()
dec["snowball (i-g)/(1+g) d"] = (dd["i_minus_g"] / (1 + dd["g"])) * dd["d"].shift(1)
dec["primary deficit"] = dd["pd"]
dec["residual"] = dec["change in d"] - dec["snowball (i-g)/(1+g) d"] - dec["primary deficit"]
(dec * 100).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
(dec.drop(columns="change in d") * 100).set_axis(
    ["Interest–growth (snowball)", "Primary deficit", "Residual"], axis=1
).plot.bar(stacked=True, ax=ax, color=[BLUE, ORANGE, LIGHT], width=0.6)
xd = np.arange(len(dec))
ax.scatter(xd, dec["change in d"] * 100, color="black", zorder=3, s=30,
           label="Change in debt ratio")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(xd, dec.index, rotation=45)
ax.set_xlabel("")
ax.set_ylabel("Percentage points of GDP")
ax.set_title("What changed the debt ratio each year?", loc="left", fontsize=13)
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Decomposition of the change in Union government "
         "liabilities/GDP. Sources: Union Budget documents; MoSPI NAS.",
         fontsize=8, color="grey")
plt.show()

## 12.10 Simulating the future: scenarios

In [ ]:
d0 = dd.loc["2026-27", "d"]
i_base = dd.loc["2026-27", "i"]
g_base = dd.loc["2026-27", "g"]
pd_base = dd.loc["2026-27", "pd"]
print(f"Starting debt ratio (BE 2026-27): {d0:.1%}")
print(f"i = {i_base:.2%}, g = {g_base:.2%}, primary deficit = {pd_base:.2%} of GDP")


def debt_path(d_start, i, g, pd, years=10):
    """Debt ratio after each of `years` years, starting from d_start."""
    path = [d_start]
    for _ in range(years):
        path.append((1 + i) / (1 + g) * path[-1] + pd)
    return np.array(path)


scenarios = {
    "Budget 2026-27 settings": (i_base, g_base, pd_base),
    "Growth 3 points lower (assumption)": (i_base, g_base - 0.03, pd_base),
    "i = g (assumption)": (i_base, i_base, pd_base),
    "Primary balance zero (assumption)": (i_base, g_base, 0.0),
}
horizon = [fy(y) for y in range(2026, 2037)]
paths = pd.DataFrame({name: debt_path(d0, *p) * 100
                      for name, p in scenarios.items()}, index=horizon)
paths.iloc[[0, 5, 10]].round(1)

In [ ]:
for name, (i, g, pd_) in scenarios.items():
    pb_star = (i - g) / (1 + g) * d0
    print(f"{name:36} pb* = {pb_star:+.2%} of GDP")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
styles = [(BLUE, "-"), (ORANGE, "-"), ("grey", "--"), (LIGHT, "-")]
for (name, col), (c, ls) in zip(paths.items(), styles):
    ax.plot(range(len(horizon)), col, color=c, linestyle=ls, linewidth=2,
            label=name)
ax.set_xticks(range(0, len(horizon), 2), horizon[::2], rotation=45)
ax.set_ylabel("Debt, % of GDP")
ax.set_title("Where is the Union debt ratio heading? Scenarios", loc="left",
             fontsize=13)
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.08, "Start: BE 2026-27. Scenario values marked 'assumption' "
         "are illustrative, not forecasts. Source: author's calculation.",
         fontsize=8, color="grey")
plt.show()

## 12.11 What this analysis leaves out

In [ ]:
state_file = Path("data/manual/rbi_state_liabilities.csv")
if state_file.exists():
    states = pd.read_csv(state_file, dtype={"year": str}).set_index("year")
    combined = (debt + states["liabilities"]).dropna()
    print("Centre + states, % of GDP (before netting intergovernmental claims):")
    print((combined / gdp.reindex(combined.index) * 100).round(1))
else:
    print("No file at", state_file, "- state finances skipped (see above).")

## 12.12 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading